### Model Evaluation

Nesta etapa avaliamos o desempenho do melhor modelo (Random Forest)
usando métricas detalhadas e análises de erro.

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col

fraud_df = spark.read.table("fraud_features")

feature_cols = [
    "amount", "transaction_hour", "foreign_transaction", 
    "location_mismatch", "device_trust_score", "velocity_last_24h",
    "cardholder_age", "high_value_transaction", "foreign_high_risk",
    "night_transaction", "amount_log", "merchant_category_index"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
fraud_df = assembler.transform(fraud_df)

# Split
train_df, test_df = fraud_df.randomSplit([0.8, 0.2], seed=42)

# Treinar RF
rf = RandomForestClassifier(featuresCol="features", labelCol="is_fraud", numTrees=100, maxDepth=10, seed=42)
rf_model = rf.fit(train_df)

# Predições
rf_predictions = rf_model.transform(test_df)
print(f"Predições criadas: {rf_predictions.count()} registros")

In [0]:
# Calcular todas as métricas
binary_eval = BinaryClassificationEvaluator(labelCol="is_fraud")
multi_eval = MulticlassClassificationEvaluator(labelCol="is_fraud", predictionCol="prediction")

auc = binary_eval.evaluate(rf_predictions, {binary_eval.metricName: "areaUnderROC"})
accuracy = multi_eval.evaluate(rf_predictions, {multi_eval.metricName: "accuracy"})
precision = multi_eval.evaluate(rf_predictions, {multi_eval.metricName: "weightedPrecision"})
recall = multi_eval.evaluate(rf_predictions, {multi_eval.metricName: "weightedRecall"})
f1 = multi_eval.evaluate(rf_predictions, {multi_eval.metricName: "f1"})

print("Métricas do Random Forest:")
print(f"AUC-ROC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")


In [0]:
# Matriz de Confusão Detalhada
print("Matriz de Confusão:")

confusion_matrix = rf_predictions.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction")
confusion_matrix.show()

# Calcular valores
tp = rf_predictions.filter((col("is_fraud") == 1) & (col("prediction") == 1.0)).count()
tn = rf_predictions.filter((col("is_fraud") == 0) & (col("prediction") == 0.0)).count()
fp = rf_predictions.filter((col("is_fraud") == 0) & (col("prediction") == 1.0)).count()
fn = rf_predictions.filter((col("is_fraud") == 1) & (col("prediction") == 0.0)).count()

total = tp + tn + fp + fn

print(f"\nTrue Positives (TP): {tp} - Fraudes corretamente identificadas")
print(f"True Negatives (TN): {tn} - Transações legítimas corretamente identificadas")
print(f"False Positives (FP): {fp} - Transações legítimas marcadas como fraude")
print(f"False Negatives (FN): {fn} - Fraudes não detectadas")
print(f"\nTotal: {total}")

In [0]:
# Métricas específicas por classe

if (tp + fp) > 0:
    precision_fraud = tp / (tp + fp)
else:
    precision_fraud = 0

if (tp + fn) > 0:
    recall_fraud = tp / (tp + fn)
else:
    recall_fraud = 0

if (precision_fraud + recall_fraud) > 0:
    f1_fraud = 2 * (precision_fraud * recall_fraud) / (precision_fraud + recall_fraud)
else:
    f1_fraud = 0

if (tn + fn) > 0:
    precision_legit = tn / (tn + fn)
else:
    precision_legit = 0

if (tn + fp) > 0:
    recall_legit = tn / (tn + fp)
else:
    recall_legit = 0

print("Métricas por Classes:")
print(f"{'Classe':<20} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print(f"{'Fraude (1)':<20} {precision_fraud:<15.4f} {recall_fraud:<15.4f} {f1_fraud:<15.4f}")
print(f"{'Legítima (0)':<20} {precision_legit:<15.4f} {recall_legit:<15.4f}")

print(f"\nTaxa de detecção de fraudes: {recall_fraud*100:.2f}%")
print(f"Taxa de falsos alarmes: {fp/total*100:.2f}%")

In [0]:
# Analisar Falso Positivos

false_positives = rf_predictions.filter((col("is_fraud") == 0) & (col("prediction") == 1.0))

print(f"Análise de Falso Positiveo (Falsos Alarmes)")
print(f"Total: {false_positives.count()}")
print(f"Percentual do total: {fp/total*100:.2f}%")

print("\nEstatísticas dos Falso Positivos:")
false_positives.select("amount", "transaction_hour", "device_trust_score", "velocity_last_24h").describe().show()

print("\nAmostra de Falso Positivos:")
false_positives.select(
    "amount", "transaction_hour", "merchant_category", 
    "foreign_transaction", "device_trust_score"
).show(10)

In [0]:
# Analisar Falso Negativeo (fraudes não detectadas)

false_negatives = rf_predictions.filter((col("is_fraud") == 1) & (col("prediction") == 0.0))

print(f"Análise de Falso Negativos (Fraudes Não Detectadas)")
print(f"Total: {false_negatives.count()}")
print(f"Percentual das fraudes: {fn/(tp+fn)*100:.2f}%")

print("\nEstatísticas dos Falso Negativos:")
false_negatives.select("amount", "transaction_hour", "device_trust_score", "velocity_last_24h").describe().show()

print("\nAmostra de Falso Negativos:")
false_negatives.select(
    "amount", "transaction_hour", "merchant_category", 
    "foreign_transaction", "device_trust_score"
).show(10)

In [0]:
# Feature Importance
feature_importance = rf_model.featureImportances.toArray()
feature_importance_list = list(zip(feature_cols, feature_importance))
feature_importance_sorted = sorted(feature_importance_list, key=lambda x: x[1], reverse=True)

print("Importância das features:")
print(f"{'Feature':<30} {'Importância':<15} {'%':<10}")

for feat, importance in feature_importance_sorted:
    print(f"{feat:<30} {importance:<15.4f} {importance*100:<10.2f}%")

In [0]:
# Resumo da Avaliação

print("Resumo")
print(f"Modelo: Random Forest (100 árvores, profundidade 10)")
print(f"\nMétricas Gerais:")
print(f"  AUC-ROC: {auc:.4f}")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  F1-Score: {f1:.4f}")

print(f"\nDetecção de Fraudes:")
print(f"  Precision: {precision_fraud:.4f} ({precision_fraud*100:.1f}%)")
print(f"  Recall: {recall_fraud:.4f} ({recall_fraud*100:.1f}%)")
print(f"  F1-Score: {f1_fraud:.4f}")

print(f"\nErros:")
print(f"  False Positives: {fp} ({fp/total*100:.2f}%)")
print(f"  False Negatives: {fn} ({fn/total*100:.2f}%)")

print("\nConclusões:")
if recall_fraud > 0.8:
    print(" Excelente capacidade de detectar fraudes")
elif recall_fraud > 0.6:
    print(" Boa capacidade de detectar fraudes")
else:
    print(" Modelo precisa melhorar detecção de fraudes")
    
if precision_fraud > 0.8:
    print(" Baixa taxa de falsos alarmes")
elif precision_fraud > 0.6:
    print(" Taxa aceitável de falsos alarmes")
else:
    print(" Alta taxa de falsos alarmes")

if f1_fraud > 0.7:
    print(" Bom equilíbrio entre precision e recall")
else:
    print(" Considerar ajustar threshold ou retreinar modelo")

In [0]:
rf_predictions.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("fraud_predictions_final")
print("Predições finais salvas como tabela permanente")
print(f"Total de registros salvos: {rf_predictions.count()}")